# Build an OSPF Knowledge Graph — in Python

A tiny, **runnable** version of the *Build a Knowledge Graph from Scratch* course.
It uses **Python + networkx** so it runs right here in Colab — no database to install.

**The one idea:** OSPF is *already* a graph — routers are nodes, adjacencies are edges,
SPF is a traversal. A *knowledge graph* is that same machine, except **you** pick the
nodes — so you can add the one node OSPF never had: the **business service** that dies
when a route does.

**Is this machine learning?** No. It is just structured facts you can question — no
training, no guessing. If you can read a routing table, you can read this.


## The five words you need

- **node** — a thing. `Area 10`, `abr-01`, `Order API`.
- **edge** — a named, directed link. `abr-01 --CONNECTS--> Area 10`.
- **label** — a node's type. `Area`, `Router`, `Service`.
- **property** — a fact stored on a node or edge. `state='down'`.
- **traversal** — walking the links to answer a question (that is the blast radius).


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# networkx and matplotlib come pre-installed in Colab.
# A DiGraph is a *directed* graph: every edge has a direction, like our arrows.
G = nx.DiGraph()
print('empty graph ready:', G)

## Step 1 — Create your first nodes

Each node gets a **label** (its type) and some **properties** (facts about it).


In [ ]:
# three OSPF areas and one border router
G.add_node('Area 0',  label='Area',   area_id='0.0.0.0', role='backbone')
G.add_node('Area 10', label='Area',   area_id='0.0.0.10')
G.add_node('Area 20', label='Area',   area_id='0.0.0.20')
G.add_node('abr-01',  label='Router', role='ABR')

print(G.number_of_nodes(), 'nodes:', list(G.nodes))

## Step 2 — Connect them (the edge carries the state)

`abr-01` reaches the backbone (Area 0) and Area 10 — but its **backbone link is down**.
We store that fact right on the edge, as a property.


In [ ]:
G.add_edge('abr-01', 'Area 0',  rel='CONNECTS', state='down')   # <-- the failure
G.add_edge('abr-01', 'Area 10', rel='CONNECTS', state='up')

for u, v, d in G.edges(data=True):
    print(f'{u} --{d["rel"]}({d["state"]})--> {v}')

## Step 3 — Tell the OSPF story

A **prefix** lives in Area 10, and a **business service** rides on that prefix.
This is the link no `show` command holds: a routing fact wired to a business fact.


In [ ]:
G.add_node('10.30.10.0/24', label='Prefix')
G.add_node('Order API',     label='Service', criticality='critical')

G.add_edge('10.30.10.0/24', 'Area 10',   rel='ORIGINATES_IN')
G.add_edge('10.30.10.0/24', 'Order API', rel='SUPPORTS')

print('graph now has', G.number_of_nodes(), 'nodes and', G.number_of_edges(), 'edges')

## See it

Colour the nodes by their label; draw the down link red and dashed.


In [ ]:
colors = {'Area': '#0f7f74', 'Router': '#3aa0ff', 'Prefix': '#9aa5ad', 'Service': '#c8702a'}
node_colors = [colors.get(G.nodes[n].get('label'), '#cccccc') for n in G]
pos = nx.spring_layout(G, seed=7)   # seed keeps the layout stable

plt.figure(figsize=(9, 6))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1800, alpha=0.92)
nx.draw_networkx_labels(G, pos, font_size=9)

down  = [(u, v) for u, v, d in G.edges(data=True) if d.get('state') == 'down']
other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in down]
nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=16, edge_color='#8894a0')
nx.draw_networkx_edges(G, pos, edgelist=down,  arrows=True, arrowsize=16, edge_color='#d34b3f', style='dashed')
nx.draw_networkx_edge_labels(G, pos, font_size=7,
    edge_labels={(u, v): d['rel'] for u, v, d in G.edges(data=True)})

plt.axis('off'); plt.title('OSPF knowledge graph'); plt.tight_layout(); plt.show()

## Step 4 — Ask it the 3 AM question

*When abr-01's backbone link is down, which services lose reachability?*

We **traverse**: a downed backbone link -> the areas that router serves ->
the prefixes that originate there -> the services those prefixes support.


In [ ]:
def blast_radius(G):
    """Services put at risk by any router whose backbone link is down."""
    at_risk = []
    for rtr, area0, d in G.edges(data=True):
        # 1) a router whose CONNECTS edge to the backbone (area 0.0.0.0) is down
        if d.get('rel') == 'CONNECTS' and d.get('state') == 'down' \
           and G.nodes[area0].get('area_id') == '0.0.0.0':
            # 2) the other, non-backbone areas that router serves
            for _, area, d2 in G.out_edges(rtr, data=True):
                if d2.get('rel') != 'CONNECTS' or G.nodes[area].get('area_id') == '0.0.0.0':
                    continue
                # 3) prefixes that originate in that area
                for pfx, _, d3 in G.in_edges(area, data=True):
                    if d3.get('rel') != 'ORIGINATES_IN':
                        continue
                    # 4) services those prefixes support
                    for _, svc, d4 in G.out_edges(pfx, data=True):
                        if d4.get('rel') == 'SUPPORTS':
                            at_risk.append((rtr, area, pfx, svc))
    return at_risk

for rtr, area, pfx, svc in blast_radius(G):
    print(f'{rtr} down  ->  {area}  ->  {pfx}  ->  AT RISK: {svc}')

The router only ever told you an adjacency dropped. Your graph just told you the
**Order API** is at risk — because you gave it the one node OSPF never had.


## What-if: repair the link, then break it again

Flip the backbone link to `up` and ask again — nothing is at risk. Set it back to
`down` — the Order API returns. You are running "what if this fails?" on a model, safely.


In [ ]:
G['abr-01']['Area 0']['state'] = 'up'      # repair
print('after repair: ', blast_radius(G) or 'nothing at risk')

G['abr-01']['Area 0']['state'] = 'down'    # break it again
print('after re-break:', [svc for *_, svc in blast_radius(G)])

## Make it yours

Change the four values below to **one** area and **one** service *you* run, then
re-run. Your own service name comes back from `blast_radius(G)` — that is the moment
the tool is yours. (Model the smallest slice that answers one real question; you can
always add one more node later.)


In [ ]:
# --- change these four values to your network ---
MY_AREA    = 'Campus Area 42'
MY_ROUTER  = 'campus-abr-01'
MY_PREFIX  = '10.20.30.0/24'
MY_SERVICE = 'Badge Service'
# ------------------------------------------------

G.add_node(MY_AREA,    label='Area',   area_id='0.0.0.42')
G.add_node(MY_ROUTER,  label='Router', role='ABR')
G.add_node(MY_PREFIX,  label='Prefix')
G.add_node(MY_SERVICE, label='Service', criticality='critical')

G.add_edge(MY_ROUTER, 'Area 0', rel='CONNECTS', state='down')   # the modelled failure
G.add_edge(MY_ROUTER, MY_AREA,  rel='CONNECTS', state='up')
G.add_edge(MY_PREFIX, MY_AREA,  rel='ORIGINATES_IN')
G.add_edge(MY_PREFIX, MY_SERVICE, rel='SUPPORTS')

for rtr, area, pfx, svc in blast_radius(G):
    print(f'{rtr} down  ->  AT RISK: {svc}')

## You built a knowledge graph

From an empty graph to a question no router can answer. The shapes you used —
*a service depends on a prefix, a prefix lives in an area, a router connects an area* —
are the same ones every bigger graph is made of.

**Where next:** the full course explains the *why* in depth, and the companion lab does
this exact thing in **Neo4j + Cypher** (a real graph database) instead of networkx —
same nodes, same edges, same blast-radius question.
